# 🚀 Custom AI Enhancer — Kaggle Cloud GPU Training Pipeline
- Model: CodeFormer Stage III CFT Fine-Tuning with ArcFace Identity Loss
- Hardware: Kaggle Nvidia T4 / P100 GPU
- Target: 2,000 Iterations + ONNX Export

In [ ]:
import os, sys, subprocess, shutil, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    try:
        c = torch.nn.Conv2d(3, 3, 3).cuda()
        x = torch.randn(1, 3, 32, 32, device='cuda')
        _ = c(x)
        print("Native CUDA Conv2D verification PASSED!")
    except Exception as e:
        print(f"CUDA check failed ({e}). Installing compatible PyTorch...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
            'torch==2.4.0+cu121', 'torchvision==0.19.0+cu121',
            '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)
        print('Compatible PyTorch installed!')
else:
    print('WARNING: Running on CPU!')


In [ ]:
# 1. Clone repository
%cd /kaggle/working
if os.path.exists('custom-ai-enhancer'):
    shutil.rmtree('custom-ai-enhancer')
!git clone https://github.com/supli6669/Enhance-Image.git custom-ai-enhancer
%cd /kaggle/working/custom-ai-enhancer

# 2. Install dependencies — pin onnxscript to a version compatible with
#    this PyTorch (the latest onnxscript dropped ParamSchema which torch.onnx needs)
!pip install -q facexlib lpips gdown onnx 'onnxscript==0.1.0.dev20231023' onnxruntime-gpu pyyaml opencv-python scikit-image
!python tools/patch_and_install_basicsr.py

# 3. Download weights and prepare dataset
!python tools/download_weights.py
# Attach a private Kaggle dataset containing the reviewed images and splits.
import os
from pathlib import Path
training_data = os.environ.get("TRAINING_DATA_DIR", "")
training_split = os.environ.get("TRAINING_SPLIT_DIR", "")
if not training_data or not training_split:
    raise RuntimeError("Set TRAINING_DATA_DIR and TRAINING_SPLIT_DIR to the attached real dataset and reviewed splits")
import subprocess, sys
subprocess.run([sys.executable, "train_custom.py", "--preflight", "--dataset-dir", training_data,
                "--split-dir", training_split], check=True)
print("Environment, weights, and dataset ready!")

In [ ]:
# Launch only after the real split and full baseline have been reviewed.
import subprocess
baseline_report = os.environ.get("BASELINE_REPORT", "")
if not baseline_report:
    raise RuntimeError("Set BASELINE_REPORT to the full evaluation JSON in the private attached dataset")
subprocess.run([sys.executable, "train_custom.py", "--fresh",
                "--dataset-dir", training_data, "--split-dir", training_split,
                "--baseline-report", baseline_report], check=True)


In [ ]:
# Select an evaluated checkpoint explicitly. Outputs remain candidates.
import subprocess
checkpoint = os.environ.get("APPROVED_CHECKPOINT", "")
if not checkpoint:
    raise RuntimeError("Set APPROVED_CHECKPOINT after evaluating the training checkpoints")
output_dir = Path("artifacts/codeformer_candidate")
output_dir.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "tools/export_onnx.py", "--checkpoint", checkpoint,
                "--output", str(output_dir / "candidate.onnx")], check=True)
subprocess.run([sys.executable, "tools/quantize_onnx_static.py", "--model", str(output_dir / "candidate.onnx"),
                "--output", str(output_dir / "candidate_int8.onnx"), "--calib-dir", training_data,
                "--calib-manifest", str(Path(training_split) / "train.txt")], check=True)
